In [0]:
%sql
CREATE OR REPLACE VIEW crm_ingestion.gold.view_customer_health_base AS (
WITH leads_processed_deduped AS (
    SELECT
    *
    FROM (
        SELECT
            id,
            status_label,
            date_updated,
            bronze_insert_date,
            ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed
        WHERE
            status_label = 'Coaching Client'
        ) 
        WHERE
            rn = 1
)
,
leads_processed_custom_deduped AS (
    SELECT
        *
    FROM (
        SELECT
            leads_processed_id,
            CSM, -- — customer success manager
            Contracted_Value, -- — sales context
            Tier, -- — account info
            Avatar, -- profile image
            bronze_insert_date,
            ROW_NUMBER() OVER (PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed_custom
    )
    WHERE
        rn = 1
)
SELECT
    l.id,
    l.status_label,
    l.date_updated,
    l.bronze_insert_date,
    NULLIF(c.CSM, '\\N') AS CSM,
    NULLIF(c.Contracted_Value, '\\N') AS Contracted_Value,
    NULLIF(c.Tier, '\\N') AS Tier,
    NULLIF(c.Avatar, '\\N') AS Avatar
FROM
    leads_processed_deduped l
LEFT JOIN
    leads_processed_custom_deduped c
ON
    l.id = c.leads_processed_id
)


In [0]:
%sql
WITH deduped_id_activities_filtered_email AS (
    SELECT
        a.id,
        a.lead_id, 
        a.type,
        a.status,
        a.activity_at,
        b.body_text,
        ROW_NUMBER() OVER (PARTITION BY a.id ORDER BY a.date_updated DESC, a.bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed a
    LEFT JOIN
        crm_ingestion.silver.lead_activites_email_body b
    ON
        a.id = b.activity_id
    WHERE 
        a.type = 'Email'
    AND a.status = 'inbox'
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%A new event has been scheduled.%')
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%is requesting access to the following folder%')
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%requests access to an item%')
),
latest_email_per_lead_id AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC) AS lead_id_rn
    FROM
        deduped_id_activities_filtered_email
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS days_since_last_email
FROM
    latest_email_per_lead_id
WHERE
    lead_id_rn = 1

In [0]:
%sql
WITH deduped_id_activities_filtered_text AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY activity_id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_email_body
    WHERE
        body_text ILIKE '%A new event has been scheduled.%'
),
latest_text_per_lead_id AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC) AS lead_id_rn
    FROM
        deduped_id_activities_filtered_text
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS days_since_last_meeting
FROM
    latest_text_per_lead_id
WHERE
    lead_id_rn = 1  

In [0]:
%sql
    
WITH deduped_id AS (
    SELECT
        id,
        lead_id,
        status,
        activity_at,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed
    WHERE
        type = 'Call'
        AND direction = 'inbound'
        AND disposition = 'no-answer'
),
deduped_calls AS (
    SELECT *
    FROM deduped_id
    WHERE id_rn = 1
)
SELECT
    lead_id,
    COUNT(*) AS missed_calls_14_days
FROM
    deduped_calls
WHERE
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) <= 14
GROUP BY
    lead_id

In [0]:
%sql
WITH deduped_id AS (
    SELECT
        id,
        lead_id,
        status,
        starts_at,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed
    WHERE
        type = 'Meeting'
        AND status = 'upcoming'
        AND DATE(starts_at) > CURRENT_DATE
),
earliest_upcoming_per_lead AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY starts_at ASC) AS lead_id_rn
    FROM
        deduped_id
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(DATE(starts_at), CURRENT_DATE) AS days_until_next_meeting
FROM
    earliest_upcoming_per_lead
WHERE
    lead_id_rn = 1

In [0]:
%sql
    
WITH coaching_clients AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
crm_to_moodle AS (
    SELECT DISTINCT
        c.leads_processed_id AS lead_id,
        m.LASTACCESS
    FROM crm_ingestion.silver.leads_processed_contacts c
    JOIN crm_ingestion.silver.mdl_users_processed m
      ON c.name = m.USERNAME
    WHERE m.LASTACCESS IS NOT NULL
      AND m.LASTACCESS != ''
    QUALIFY ROW_NUMBER() OVER (PARTITION BY c.leads_processed_id ORDER BY m.LASTACCESS DESC) = 1
)
SELECT
    cc.lead_id,
    DATEDIFF(CURRENT_DATE, DATE(m.LASTACCESS)) AS days_since_last_login
FROM coaching_clients cc
JOIN crm_to_moodle m ON cc.lead_id = m.lead_id

In [0]:
%sql
    
WITH coaching_clients AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
calendly_events AS (
    SELECT
        INVITEE_NAME,
        EVENT_START_TIME
    FROM crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE INVITEE_NAME IS NOT NULL
      AND EVENT_START_TIME IS NOT NULL
      AND DATE(EVENT_START_TIME) < CURRENT_DATE
    QUALIFY ROW_NUMBER() OVER (PARTITION BY EVENT_URI ORDER BY INVITEE_UPDATED_AT DESC) = 1
),
contacts_bridge AS (
    SELECT DISTINCT
        lc.name,
        lc.leads_processed_id AS lead_id
    FROM crm_ingestion.silver.leads_processed_contacts lc
    WHERE lc.name IS NOT NULL
),
calendly_matched AS (
    SELECT
        cb.lead_id,
        ce.EVENT_START_TIME
    FROM contacts_bridge cb
    JOIN calendly_events ce ON ce.INVITEE_NAME = cb.name
    QUALIFY ROW_NUMBER() OVER (PARTITION BY cb.lead_id ORDER BY ce.EVENT_START_TIME DESC) = 1
)
SELECT
    cc.lead_id,
    DATEDIFF(CURRENT_DATE, DATE(cm.EVENT_START_TIME)) AS days_since_last_meeting_calendly
FROM coaching_clients cc
JOIN calendly_matched cm ON cc.lead_id = cm.lead_id

In [0]:
%sql
-- Sentiment + message recency per lead_id
-- Data originates from student_sentiment source, already joined to leads by CRM
-- and stored in leads_processed_custom. We clean \N → NULL and exclude C082U471YMU.
SELECT
    leads_processed_id AS lead_id,
    NULLIF(SENTIMENTS_LAST_30_DAYS, '\\N') AS sentiment,
    TRY_CAST(DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT AS BIGINT) AS days_since_last_message_from_client,
    TRY_CAST(DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER AS BIGINT) AS days_since_last_message_from_team_member
FROM (
    SELECT
        leads_processed_id,
        Channel_ID,
        SENTIMENTS_LAST_30_DAYS,
        DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
        DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
        ROW_NUMBER() OVER(PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) AS rn
    FROM
        crm_ingestion.silver.leads_processed_custom
    WHERE
        Channel_ID IS NOT NULL
        AND Channel_ID != ''
        AND Channel_ID != 'C082U471YMU'  -- excluded channel (spec 5.1)
)
WHERE rn = 1

In [0]:
%sql
-- ============================================================
-- CUSTOMER HEALTH PROFILE — Final Table
-- Combines all 7 metrics + sentiment + sales context + scoring
-- Creates: crm_ingestion.gold.customer_health_profile
-- Spec: one current profile per source LEAD_ID, lookup by email
-- ============================================================

CREATE OR REPLACE TABLE crm_ingestion.gold.customer_health_profile AS
WITH
-- --- Metric 1: DAYS_SINCE_LAST_EMAIL ---
deduped_email_activities AS (
    SELECT
        a.id,
        a.lead_id,
        a.type,
        a.status,
        a.activity_at,
        b.body_text,
        ROW_NUMBER() OVER (PARTITION BY a.id ORDER BY a.date_updated DESC, a.bronze_insert_date DESC) AS id_rn
    FROM crm_ingestion.silver.lead_activites_processed a
    LEFT JOIN crm_ingestion.silver.lead_activites_email_body b
      ON a.id = b.activity_id
    WHERE a.type = 'Email'
      AND a.status = 'inbox'
      AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%A new event has been scheduled.%')
      AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%is requesting access to the following folder%')
      AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%requests access to an item%')
),
latest_email_per_lead AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC) AS lead_id_rn
    FROM deduped_email_activities
    WHERE id_rn = 1
),
calc_email AS (
    SELECT
        lead_id,
        DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS calc_days_since_last_email
    FROM latest_email_per_lead
    WHERE lead_id_rn = 1
),

-- --- Metric 2: DAYS_SINCE_LAST_MEETING ---
deduped_meeting_text AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY activity_id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM crm_ingestion.silver.lead_activites_email_body
    WHERE body_text ILIKE '%A new event has been scheduled.%'
),
latest_meeting_text_per_lead AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC) AS lead_id_rn
    FROM deduped_meeting_text
    WHERE id_rn = 1
),
calc_meeting AS (
    SELECT
        lead_id,
        DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS calc_days_since_last_meeting
    FROM latest_meeting_text_per_lead
    WHERE lead_id_rn = 1
),

-- --- Metric 3: MISSED_CALLS_14_DAYS ---
deduped_missed_calls AS (
    SELECT
        id,
        lead_id,
        status,
        activity_at,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM crm_ingestion.silver.lead_activites_processed
    WHERE type = 'Call'
      AND direction = 'inbound'
      AND disposition = 'no-answer'
),
calc_missed_calls AS (
    SELECT
        lead_id,
        COUNT(*) AS calc_missed_calls_14_days
    FROM deduped_missed_calls
    WHERE id_rn = 1
      AND DATEDIFF(CURRENT_DATE, DATE(activity_at)) <= 14
    GROUP BY lead_id
),

-- --- Metric 4: DAYS_UNTIL_NEXT_MEETING ---
deduped_upcoming AS (
    SELECT
        id,
        lead_id,
        status,
        starts_at,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM crm_ingestion.silver.lead_activites_processed
    WHERE type = 'Meeting'
      AND status = 'upcoming'
      AND DATE(starts_at) > CURRENT_DATE
),
earliest_upcoming_per_lead AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY starts_at ASC) AS lead_id_rn
    FROM deduped_upcoming
    WHERE id_rn = 1
),
calc_upcoming AS (
    SELECT
        lead_id,
        DATEDIFF(DATE(starts_at), CURRENT_DATE) AS calc_days_until_next_meeting
    FROM earliest_upcoming_per_lead
    WHERE lead_id_rn = 1
),

-- --- Metric 5: DAYS_SINCE_LAST_LOGIN ---
coaching_clients_login AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
crm_to_moodle AS (
    SELECT DISTINCT
        c.leads_processed_id AS lead_id,
        m.LASTACCESS
    FROM crm_ingestion.silver.leads_processed_contacts c
    JOIN crm_ingestion.silver.mdl_users_processed m
      ON c.name = m.USERNAME
    WHERE m.LASTACCESS IS NOT NULL
      AND m.LASTACCESS != ''
    QUALIFY ROW_NUMBER() OVER (PARTITION BY c.leads_processed_id ORDER BY m.LASTACCESS DESC) = 1
),
calc_login AS (
    SELECT
        cc.lead_id,
        DATEDIFF(CURRENT_DATE, DATE(m.LASTACCESS)) AS calc_days_since_last_login
    FROM coaching_clients_login cc
    JOIN crm_to_moodle m ON cc.lead_id = m.lead_id
),

-- --- Metric 6: DAYS_SINCE_LAST_MEETING_CALENDLY ---
coaching_clients_cal AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
calendly_events AS (
    SELECT
        INVITEE_NAME,
        EVENT_START_TIME
    FROM crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE INVITEE_NAME IS NOT NULL
      AND EVENT_START_TIME IS NOT NULL
      AND DATE(EVENT_START_TIME) < CURRENT_DATE
    QUALIFY ROW_NUMBER() OVER (PARTITION BY EVENT_URI ORDER BY INVITEE_UPDATED_AT DESC) = 1
),
contacts_bridge AS (
    SELECT DISTINCT
        lc.name,
        lc.leads_processed_id AS lead_id
    FROM crm_ingestion.silver.leads_processed_contacts lc
    WHERE lc.name IS NOT NULL
),
calendly_matched AS (
    SELECT
        cb.lead_id,
        ce.EVENT_START_TIME
    FROM contacts_bridge cb
    JOIN calendly_events ce ON ce.INVITEE_NAME = cb.name
    QUALIFY ROW_NUMBER() OVER (PARTITION BY cb.lead_id ORDER BY ce.EVENT_START_TIME DESC) = 1
),
calc_calendly AS (
    SELECT
        cc.lead_id,
        DATEDIFF(CURRENT_DATE, DATE(cm.EVENT_START_TIME)) AS calc_days_since_last_meeting_calendly
    FROM coaching_clients_cal cc
    JOIN calendly_matched cm ON cc.lead_id = cm.lead_id
),

-- --- Metric 7: SENTIMENT + MESSAGE RECENCY ---
calc_sentiment AS (
    SELECT
        leads_processed_id AS lead_id,
        NULLIF(SENTIMENTS_LAST_30_DAYS, '\\N') AS sentiment,
        TRY_CAST(DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT AS BIGINT) AS days_since_last_message_from_client,
        TRY_CAST(DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER AS BIGINT) AS days_since_last_message_from_team_member
    FROM (
        SELECT
            leads_processed_id,
            Channel_ID,
            SENTIMENTS_LAST_30_DAYS,
            DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
            DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
            ROW_NUMBER() OVER(PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed_custom
        WHERE
            Channel_ID IS NOT NULL
            AND Channel_ID != ''
            AND Channel_ID != 'C082U471YMU'
    )
    WHERE rn = 1
),

-- --- Lead names for profile ---
lead_info AS (
    SELECT id AS lead_id, name AS lead_name, display_name AS lead_display_name
    FROM (
        SELECT id, name, display_name,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
        FROM crm_ingestion.silver.leads_processed
        WHERE status_label = 'Coaching Client'
    )
    WHERE rn = 1
),

-- --- Lead emails for lookup ---
lead_emails AS (
    SELECT
        leads_processed_id AS lead_id,
        collect_set(email) AS customer_emails
    FROM crm_ingestion.silver.leads_processed_contacts_emails
    WHERE email IS NOT NULL AND email != ''
    GROUP BY leads_processed_id
),

-- --- Sales context from Gold ---
sales AS (
    SELECT * FROM crm_ingestion.gold.sales_details
),

-- --- CSM name resolution ---
csm_names AS (
    SELECT user_key, full_name AS csm_name, email_normalized AS csm_email
    FROM crm_ingestion.gold.close_crm_users_processed
),

-- --- FINAL JOIN: all metrics + sentiment + lead info + emails + sales ---
health_metrics AS (
    SELECT
        b.id AS lead_id,
        b.status_label,
        b.date_updated,
        li.lead_name,
        li.lead_display_name,
        le.customer_emails,
        b.CSM,
        csm.csm_name,
        csm.csm_email,
        b.Contracted_Value,
        b.Tier,
        b.Avatar,
        sd.date_of_sale,
        sd.program,
        sd.setter_name,
        sd.closer_name,
        sd.contract_value AS sale_contract_value,
        e.calc_days_since_last_email,
        m.calc_days_since_last_meeting,
        COALESCE(mc.calc_missed_calls_14_days, 0) AS missed_calls_14_days,
        nm.calc_days_until_next_meeting,
        l.calc_days_since_last_login,
        cl.calc_days_since_last_meeting_calendly,
        s.sentiment,
        s.days_since_last_message_from_client,
        s.days_since_last_message_from_team_member
    FROM crm_ingestion.gold.view_customer_health_base b
    LEFT JOIN lead_info li ON b.id = li.lead_id
    LEFT JOIN lead_emails le ON b.id = le.lead_id
    LEFT JOIN sales sd ON b.id = sd.lead_id
    LEFT JOIN csm_names csm ON b.CSM = csm.user_key
    LEFT JOIN calc_email e ON b.id = e.lead_id
    LEFT JOIN calc_meeting m ON b.id = m.lead_id
    LEFT JOIN calc_missed_calls mc ON b.id = mc.lead_id
    LEFT JOIN calc_upcoming nm ON b.id = nm.lead_id
    LEFT JOIN calc_login l ON b.id = l.lead_id
    LEFT JOIN calc_calendly cl ON b.id = cl.lead_id
    LEFT JOIN calc_sentiment s ON b.id = s.lead_id
),

-- --- Engagement flags (spec 5.1 hardcoded thresholds) ---
flags AS (
    SELECT
        h.*,
        CASE
            WHEN h.days_since_last_message_from_client IS NULL THEN NULL
            WHEN h.days_since_last_message_from_client <= 14 THEN 1
            ELSE 0
        END AS slack_engaged_flag,
        CASE
            WHEN h.calc_days_since_last_login IS NULL THEN NULL
            WHEN h.calc_days_since_last_login <= 14 THEN 1
            ELSE 0
        END AS platform_engaged_flag,
        CASE
            WHEN h.calc_days_since_last_meeting IS NULL OR h.calc_days_since_last_meeting_calendly IS NULL THEN NULL
            WHEN h.calc_days_since_last_meeting <= 30 OR h.calc_days_since_last_meeting_calendly <= 30 THEN 1
            ELSE 0
        END AS meeting_engaged_flag
    FROM health_metrics h
),

-- --- Engagement pattern + adjustments (spec 5.1 ordered rules) ---
patterns AS (
    SELECT
        f.*,
        CASE
            WHEN f.slack_engaged_flag = 1 AND f.platform_engaged_flag = 1 AND f.meeting_engaged_flag = 1 THEN 'SLACK + PLATFORM + MEETING'
            WHEN f.slack_engaged_flag = 1 AND f.platform_engaged_flag = 1 AND (f.meeting_engaged_flag = 0 OR f.meeting_engaged_flag IS NULL) THEN 'SLACK + PLATFORM'
            WHEN (f.slack_engaged_flag = 0 OR f.slack_engaged_flag IS NULL) AND f.platform_engaged_flag = 1 AND f.meeting_engaged_flag = 1 THEN 'PLATFORM + MEETING'
            WHEN f.slack_engaged_flag = 1 AND (f.platform_engaged_flag = 0 OR f.platform_engaged_flag IS NULL) AND f.meeting_engaged_flag = 1 THEN 'SLACK + MEETING'
            WHEN f.slack_engaged_flag = 1 AND (f.platform_engaged_flag = 0 OR f.platform_engaged_flag IS NULL) AND (f.meeting_engaged_flag = 0 OR f.meeting_engaged_flag IS NULL) THEN 'SLACK ONLY'
            WHEN (f.slack_engaged_flag = 0 OR f.slack_engaged_flag IS NULL) AND f.platform_engaged_flag = 1 AND (f.meeting_engaged_flag = 0 OR f.meeting_engaged_flag IS NULL) THEN 'PLATFORM ONLY'
            WHEN (f.slack_engaged_flag = 0 OR f.slack_engaged_flag IS NULL) AND (f.platform_engaged_flag = 0 OR f.platform_engaged_flag IS NULL) AND f.meeting_engaged_flag = 1 THEN 'MEETING ONLY'
            WHEN f.slack_engaged_flag = 0 AND f.platform_engaged_flag = 0 AND f.meeting_engaged_flag = 0 THEN 'NO ENGAGEMENT'
            WHEN f.slack_engaged_flag = 0 AND f.platform_engaged_flag = 0 AND f.meeting_engaged_flag IS NULL THEN 'NO ENGAGEMENT'
            WHEN f.slack_engaged_flag IS NULL AND f.platform_engaged_flag = 0 AND f.meeting_engaged_flag = 0 THEN 'NO ENGAGEMENT'
            WHEN f.slack_engaged_flag = 0 AND f.platform_engaged_flag IS NULL AND f.meeting_engaged_flag = 0 THEN 'NO ENGAGEMENT'
            ELSE 'NO DATA'
        END AS engagement_pattern,
        CASE
            WHEN f.missed_calls_14_days = 0 THEN 5
            WHEN f.missed_calls_14_days BETWEEN 1 AND 2 THEN 0
            WHEN f.missed_calls_14_days >= 3 THEN -15
        END AS missed_call_adjustment,
        CASE
            WHEN UPPER(TRIM(f.sentiment)) = 'POSITIVE' THEN 10
            WHEN UPPER(TRIM(f.sentiment)) = 'NEUTRAL' THEN 5
            WHEN UPPER(TRIM(f.sentiment)) = 'NEGATIVE' THEN -25
            ELSE 0
        END AS sentiment_adjustment
    FROM flags f
),

-- --- Base score from engagement pattern ---
scores AS (
    SELECT
        p.*,
        CASE p.engagement_pattern
            WHEN 'SLACK + PLATFORM + MEETING' THEN 90
            WHEN 'SLACK + PLATFORM' THEN 80
            WHEN 'PLATFORM + MEETING' THEN 75
            WHEN 'SLACK + MEETING' THEN 60
            WHEN 'PLATFORM ONLY' THEN 50
            WHEN 'MEETING ONLY' THEN 40
            WHEN 'SLACK ONLY' THEN 30
            WHEN 'NO ENGAGEMENT' THEN 20
            ELSE NULL
        END AS base_score
    FROM patterns p
)

-- --- FINAL OUTPUT ---
SELECT
    s.lead_id,
    s.status_label,
    s.lead_name,
    s.lead_display_name,
    s.customer_emails,
    s.CSM,
    s.csm_name,
    s.csm_email,
    s.Contracted_Value,
    s.Tier,
    s.Avatar,
    s.date_of_sale,
    s.program,
    s.setter_name,
    s.closer_name,
    s.sale_contract_value,
    s.calc_days_since_last_email,
    s.calc_days_since_last_meeting,
    s.missed_calls_14_days,
    s.calc_days_until_next_meeting,
    s.calc_days_since_last_login,
    s.calc_days_since_last_meeting_calendly,
    COALESCE(UPPER(TRIM(s.sentiment)), 'NO_SENTIMENT') AS sentiment,
    s.days_since_last_message_from_client,
    s.days_since_last_message_from_team_member,
    s.slack_engaged_flag,
    s.platform_engaged_flag,
    s.meeting_engaged_flag,
    s.engagement_pattern,
    s.base_score,
    s.missed_call_adjustment,
    s.sentiment_adjustment,
    CASE
        WHEN s.engagement_pattern = 'NO DATA' THEN NULL
        ELSE s.base_score + s.missed_call_adjustment + s.sentiment_adjustment
    END AS final_health_score,
    CASE
        WHEN s.engagement_pattern = 'NO DATA' THEN 'NO DATA'
        WHEN s.base_score + s.missed_call_adjustment + s.sentiment_adjustment >= 75 THEN 'GOOD'
        WHEN s.base_score + s.missed_call_adjustment + s.sentiment_adjustment >= 50 THEN 'AVERAGE'
        ELSE 'POOR'
    END AS health_band
FROM scores s
